# Transform Metadata Demo Using Client Library

This notebook demonstrates how to use the Curator workflows client library to transform tables to match target schemas using AI-powered column translation.

## Setup and Configuration

In [ ]:
import sys
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Import the client library
sys.path.append('../../src')
from transform_client import TransformMetadataClient

# Load environment variables
load_dotenv()

# Create client
client = TransformMetadataClient()

print(f"API URL: {client.api_url}")
print(f"API Key configured: {'Yes' if client.api_key else 'No'}")

## Transform Metadata

The transform_metadata function processes CSV files to transform them into a target schema using AI-powered column translation. The workflow:

1. Request presigned URL and upload the source CSV
2. Monitor progress until completion
3. Download and display the transformed table matching the target schema

In [ ]:
# Path to the source CSV file to be transformed
csv_file_path = "data/transform_metadata_source.csv"

# Check if file exists
if not os.path.exists(csv_file_path):
    print(f"❌ CSV file not found at: {csv_file_path}")
    print("📝 Note: Create a source CSV file and place it in the data/ directory")
    print("Example source CSV content:")
    print("first_name,last_name,email_address,phone_number")
    print("John,Doe,john.doe@email.com,555-1234")
    print("Jane,Smith,jane.smith@email.com,555-5678")
    print("Bob,Johnson,bob.johnson@email.com,555-9012")
else:
    print(f"📄 Found CSV file: {csv_file_path}")
    print(f"File size: {os.path.getsize(csv_file_path)} bytes")

In [ ]:
# Define a callback function for status updates
def status_callback(status_data, attempt, max_attempts):
    print(f"\n📊 Attempt {attempt}/{max_attempts}...")
    
    # Print basic status information
    status = status_data.get('status', 'UNKNOWN')
    print(f"Status: {status}")
    
    # Print detailed execution information if available
    execution_status = status_data.get('executionStatus', {})
    if execution_status:
        print(f"Current state: {execution_status.get('currentState', 'Unknown')}")
        print(f"Execution status: {execution_status.get('status', 'Unknown')}")

In [ ]:
# Transform the CSV file to match the target schema
print("🔍 Transforming CSV file to match target schema...")

result = client.transform_metadata(
    file_path=csv_file_path,
    schema_name="transform_metadata_target_schema",
    max_attempts=30,
    interval=10,
    output_path="transformed_metadata.csv",
    callback=status_callback
)

In [ ]:
print(result)

In [ ]:
# Display the results
if result.get('status').get('executionStatus').get('output'):
    print(f"\n✅ Transformed data saved to: {result.get('status').get('executionStatus').get('output').get('translated_table_path')}")
    print("\nTransformed data preview:")
    data_json = json.loads(result.get('status', {}).get('executionStatus', {}).get('output', {}).get('data'))
    df = pd.DataFrame(data_json)
    display(df)
    
    # Display transformation details if available
    if result.get('status', {}).get('executionStatus', {}).get('output', {}).get('stats', {}):
        statistics = result.get('status', {}).get('executionStatus', {}).get('output', {}).get('stats', {})
        print("📊 Transformation Statistics:")
        for key, value in statistics.items():
            print(f"  {key.replace('_', ' ').title()}: {value}")

else:
    print("\n❌ Failed to transform metadata")
    if "error" in result:
        print(f"Error: {result['error']}")

## Understanding the Transform Metadata Process

The transform_metadata workflow consists of three main steps:

1. **Create Column Translation Tasks**: Analyzes the target schema and creates translation tasks for each target column
2. **Translate Columns**: Uses AI to generate SQL operations for creating each target column from source columns
3. **Consolidate Columns**: Applies all SQL operations to the source table to create the final transformed table

The transformed table will be available in S3 and can be downloaded using the appropriate S3 paths provided in the final status response.

### Sample Target Schema Format

The target schema file should define the desired output table structure:

```csv
full_name,contact_email,phone,team,start_date
,,,"Engineering",
,,,"Sales",
,,,"IT",
,,,"HR",
,,,"Other",
```

### Sample Source CSV

```csv
first_name,last_name,email_address,phone_number
John,Doe,john.doe@email.com,555-1234
Jane,Smith,jane.smith@email.com,555-5678
Bob,Johnson,bob.johnson@email.com,555-9012
```

### Expected Transformed CSV

```csv
full_name,contact_email,phone
John Doe,john.doe@email.com,555-1234
Jane Smith,jane.smith@email.com,555-5678
Bob Johnson,bob.johnson@email.com,555-9012
```

### AI-Generated SQL Operations

The AI will generate SQL operations like:
- `full_name`: `CONCAT(first_name, ' ', last_name)`
- `contact_email`: `email_address` (direct mapping)
- `phone`: `phone_number` (direct mapping)

In [ ]:
# If we have both original and transformed data, we can compare them
if result.get('status').get('executionStatus').get('output').get('translated_table_path') and os.path.exists(csv_file_path):
    # Read the original data
    original_data = pd.read_csv(csv_file_path)
    transformed_data = pd.DataFrame(json.loads(result.get('status', {}).get('executionStatus', {}).get('output', {}).get('data')))
    
    # Display side-by-side comparison for a few rows
    print("\nSide-by-side comparison of original and transformed data:")
    
    # Display original data
    print("\nOriginal data:")
    display(original_data.head())
    
    # Display transformed data
    print("\nTransformed data:")
    display(transformed_data.head())